# 文字转语音（TTS）

由于安全方面的原因，你并不能通过 JupyterLab 来直接访问音频设备（环境的限制），我们这里的代码块不供用户运行。

这里的程序来自于产品主程序的 audio_ctrl.py，你可以参考这里的代码来了解产品主程序是如何执行文字转语音功能的。

In [ ]:
import pyttsx3  # 导入 pyttsx3 库，用于本地 TTS（文本转语音）
import threading  # 导入 threading 模块，用于多线程处理
import numpy as np  # 导入 NumPy，用于处理音频数据
import sherpa_onnx  # 导入 sherpa_onnx，用于加载离线 TTS 模型
import soundfile as sf  # 导入 soundfile，用于保存音频文件
import re  # 导入正则库，用于判断字符串中是否包含中文

# 初始化 pyttsx3 引擎（用于播放非中文的 TTS）
engine = pyttsx3.init()

# 创建一个线程事件对象，用于控制语音播放的同步，避免同时播放多段音频
play_audio_event = threading.Event()

# 设置 TTS 播放的语速（180 表示语速稍快）
engine.setProperty('rate', 180)

# 配置 sherpa-onnx 离线 TTS 模型（中文 VITS 模型）
tts_config = sherpa_onnx.OfflineTtsConfig(
    model=sherpa_onnx.OfflineTtsModelConfig(
        vits=sherpa_onnx.OfflineTtsVitsModelConfig(
            model='./models/sherpa-onnx-vits-zh-ll/model.onnx',  # 模型文件
            lexicon='./models/sherpa-onnx-vits-zh-ll/lexicon.txt',  # 词典文件
            data_dir='',
            dict_dir='./models/sherpa-onnx-vits-zh-ll/dict',  # 拼音词典
            tokens='./models/sherpa-onnx-vits-zh-ll/tokens.txt',  # token 文件
        ),
        matcha=sherpa_onnx.OfflineTtsMatchaModelConfig(),  # 其他 TTS 模型占位
        kokoro=sherpa_onnx.OfflineTtsKokoroModelConfig(),  # 其他 TTS 模型占位
        provider='cpu',  # 使用 CPU 推理
        debug=False,
        num_threads=2,  # 推理线程数
    ),
    rule_fsts='./models/sherpa-onnx-vits-zh-ll/number.fst',  # 数字转写规则
    max_num_sentences=1,  # 每次只处理一句
)

# 初始化离线 TTS 引擎
tts = sherpa_onnx.OfflineTts(tts_config)


def play_audio(input_audio_file):
    """播放指定的音频文件"""
    if not usb_connected:
        return
    try:
        pygame.mixer.music.load(input_audio_file)  # 加载音频文件
        pygame.mixer.music.play()  # 播放音频
    except:
        play_audio_event.clear()
        return
    while pygame.mixer.music.get_busy():  # 等待播放完成
        pass
    time.sleep(min_time_bewteen_play)  # 播放间隔
    play_audio_event.clear()


def contains_chinese(text):
    """判断文本中是否包含中文字符"""
    return bool(re.search('[\u4e00-\u9fff]', text))


def play_speech(input_text):
    """将文本转换为语音并播放"""
    filename = 'audio-say.wav'
    if not usb_connected:
        return

    try:
        if contains_chinese(input_text):
            # 中文走 VITS 模型
            audio = tts.generate(input_text, sid=4, speed=1.0)
            scale = 4  # 放大音量
            samples = np.array(audio.samples) * scale
            samples = np.clip(samples, -1.0, 1.0)  # 限制音量范围

            # 保存为 wav 文件
            sf.write(filename, samples, samplerate=audio.sample_rate, subtype="PCM_16")

            # 等待文件生成
            for _ in range(10):
                if os.path.exists(filename) and os.path.getsize(filename) > 0:
                    break
                time.sleep(0.1)

            play_audio(filename)  # 播放音频文件
        else:
            # 非中文直接用 pyttsx3 播放
            engine.say(input_text)
            engine.runAndWait()

    except Exception as e:
        print(f"[play failure] {e}")
    finally:
        # 播放完成后删除临时文件
        if os.path.exists(filename):
            try:
                os.remove(filename)
            except Exception as e:
                print(f"[delete file failure] {e}")
        play_audio_event.clear()


def play_speech_thread(input_text):
    """在新线程中播放语音，避免阻塞主线程"""
    if play_audio_event.is_set():  # 如果已有播放任务则忽略
        return
    play_audio_event.set()  # 标记开始播放
    speech_thread = threading.Thread(target=play_speech, args=(input_text,))
    speech_thread.start()  # 启动新线程

这段代码使用了 sherpa_onnx，pyttsx3 库来实现文本转语音的功能，并使用 threading 模块创建了一个线程来异步播放语音。play_speech() 函数用于在主线程中播放指定文本的语音，而 play_speech_thread() 函数用于在新线程中播放语音，以避免阻塞主线程。同时，通过 play_audio_event 控制语音播放的同步，确保同一时间只有一个语音在播放。